## 0. Import necessary library.

In [1]:
import os, sys, time, torch, shutil
from pathlib import Path
from imagegrains import grainsizing, plotting, segmentation_helper, data_loader, gsd_uncertainty
from cellpose import models, io
import pandas as pd
from natsort import natsorted
import matplotlib.pyplot as plt
from PIL import Image      
import numpy as np   
from skimage.color import label2rgb
from skimage.segmentation import find_boundaries
from pathlib import Path
import imagegrains
CodeDir = Path.cwd().parent
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()        

## 1. Define working directory for model and data.

In [2]:
ts0 = time.time()
DataDir = ''
WorkDir = Path(DataDir) if DataDir and Path(DataDir).is_dir() else Path(CodeDir) / "demo_data"
WorkDir = str(WorkDir)

# Parameters.
diameter= None          # Mean grain diameter in pixels to rescale images to; default is None, which leads to automated size estimation.
second_diameter = None  # Enables a two-step segmentation that will combine predictions at two different scales. It will use `diameter` (default: None) and expects the `second_diameter` to be >> `diamter`. Predictions are combined on a pixel basis.'
threshold = 150         # If predictions from two scales are combined, this threshold (default=150) defines the cutoff above which grains from predictions with `second_diameter` will be used. Larger grains are prioritized.
do_composites = False   # Save a composite of all images and segmentation masks as .png files.
max_size_fraction = 1   # Masks larger than max_size_fraction of total image size are removed.(default=0.4) defines the cutoff above which grains from predictions with `second_diameter` will be used. Larger grains are prioritized.
pixel_cutoff = 12       # Minimum object diameter in pixels to segement; default is 12 pixels based on https://doi.org/10.5194/egusphere-2025-6346, 2026.
edge_cutoff = 0.0       # Filter by edge (imagegrains/src/imagegrains/grainsizing.py line 636) w(edge~1-edge), l*(edge,1-edge); default is 0.1.
overwrite_seg = True    # If True, overwrite segmentation mask generation.
overwrite_plot = False  # If True, overwrite plot generation.
gpu = True              # If True, use GPU for computations.
computer = 'H100'       # Computer type indicator.
dpi = 600               # Image resolution for saved overlay and masks.
model_lists = ['IG2_full_set_cp_SAM'] # Best model used in https://doi.org/10.5194/egusphere-2025-6346, 2026.

In [3]:
casenames0 = []
casenames1 = [d for d in os.listdir(WorkDir) if os.path.isdir(os.path.join(WorkDir, d))]
casenames = list(dict.fromkeys(casenames0 + casenames1))
img_paths = natsorted([os.path.join(WorkDir, case) for case in casenames])
print("All data paths to process:")
for path in img_paths:
    print(os.path.basename(path))

All data paths to process:
FH


## 2. Download and load pre-trained models. 

In [4]:
ts1 = time.time()
model_dir = os.path.join(CodeDir,'models')
if not os.path.isdir(model_dir) or not os.listdir(model_dir):
    # Download the model
    os.makedirs(model_dir, exist_ok=True)
    model_id = model_lists[0]
    model_path = os.path.join(model_dir, model_id)
    model_url = 'https://zenodo.org/records/15728186/files/'
    source_path = f'{model_url}{model_id}'
    print(f"No model found in the directory {model_dir}")
    print(f"Downloading model from {source_path} to {model_path}")
    data_loader.download_url_to_file(source_path, model_path)
model_id = os.listdir(model_dir)[0]
model_path = os.path.join(model_dir, model_id)                            # Load the first model in the directory

#initialiize the models
print(f"Loading model from {model_path}")
model = models.CellposeModel(gpu=gpu, pretrained_model=model_path)
te1 = time.time() - ts1
print(f"Loading model in {te1:.2f} seconds")

# Print model information
print("\n=== Model Information ===")
print(f"Model ID: {model_id}")
print(f"GPU: {gpu}")

# Count total parameters
if hasattr(model, 'net'):
    total_params = sum(p.numel() for p in model.net.parameters())
    trainable_params = sum(p.numel() for p in model.net.parameters() if p.requires_grad)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Model Size: {total_params * 4 / (1024**2):.2f} MB")  # Assuming float32
    # Print model architecture
    # print(f"\nModel Architecture:")
    #print(model.net)
else:
    print("Model network not accessible")

Loading model from /qfs/people/chen200/Computation/Python/work/Mine/SFA/imagegrains/models/IG2_full_set_cp_SAM


Loading model in 2.26 seconds

=== Model Information ===
Model ID: IG2_full_set_cp_SAM
GPU: True
Total Parameters: 304,628,930
Trainable Parameters: 304,592,064
Model Size: 1162.07 MB


## 3. Complete ImageGrains analysis

### 3a. Segment all images in all folders using AI model.

In [5]:
ts2 = time.time()
seg_savefolder = 'segments_ig2'
parent_folder = os.path.basename(WorkDir)
csvname_all = WorkDir + os.sep + 'Time_V3_' + parent_folder + '_' + model_id + '_' + computer + '.csv'
colnames = ['Name', 'Folder', 'Model_ID', 'Model_File', 'Width_pixel',\
       'Height_pixel', 'Imagereading_time_seconds', 'Predict_time_seconds',\
       'Preprocess_time_seconds', 'Inference_time_seconds',\
       'Postprocess_time_seconds', 'Total_time_seconds']

# Potential parameters available for config;
config = {"max_size_fraction":max_size_fraction}
filters= {'edge':[True,edge_cutoff],'px_cutoff':[True,pixel_cutoff]}
sts_all = pd.DataFrame(columns=colnames)
for im_path in img_paths:
    print('=='*65)
    ts4t = 0
    ts5t = 0
    sts = []
    foldername = os.path.basename(os.path.normpath(im_path))    
    csvname = im_path + os.sep + 'Time_V3_' + foldername + '_' + model_id + '_' + computer + '.csv'
    print(f"Processing folder: {foldername}")
    if os.path.isfile(csvname) and not overwrite_seg:
        print(f"Existing time file {csvname} found, skipping...")
        sts_df = pd.read_csv(csvname)
    else:
        # Identify all image files in the folder to process.
        img_files = natsorted([f for f in os.listdir(im_path) if f.lower().endswith(('.jpg', '.png'))])
        npi = len(img_files)                                     # Number of input images in the folder.

        # Identify already processed images.
        pred_dir = os.path.join(im_path,seg_savefolder)
        pred_dir1 = os.path.join(pred_dir,'first_diameter')
        pred_dir2 = os.path.join(pred_dir,'second_diameter')
        npp = 0
        if os.path.isdir(pred_dir):
            pred_files = natsorted([f for f in os.listdir(pred_dir) if f.endswith(model_id + '_pred.tif')])
            npp = len(pred_files) 
        
        # Checking if all images have been processed and skipping if so.
        if npp == npi and not overwrite_seg:
            print(f"{im_path}: all image predictions found, skipping....")
        else:
            for i in range(npi):
                img_file = img_files[i]
                img_stem = os.path.splitext(img_file)[0]
                im_path_s = os.path.join(im_path, img_file)
                im_path_p = os.path.join(pred_dir, img_stem + f'_{model_id}_pred.tif')

                # Get image width and height without loading image
                with Image.open(im_path_s) as _im: sw, sh = _im.size
                
                # Checking if prediction has been made for this image and skipping if so.
                te4 = 0
                te5 = 0
                if os.path.isfile(im_path_p) and not overwrite_seg:
                    print(f"{img_file}, {i+1}/{npi}, prediction found, skipping ({te4:.2f} seconds)....")
                else:
                    te4a, te4b, te4c, te4d = 0, 0, 0, 0
                    te5a, te5b = 0, 0
                    te6 = 0

                    # Default parameters.
                    ts5a = time.time()
                    io.imread(str(im_path_s))
                    te5a = time.time() - ts5a

                    ts4a = time.time()
                    segmentation_helper.predict_single_image(
                        im_path_s,diameter=diameter,model=model,config=config,
                        save_masks=True,mute =True,model_id=model_id,tar_dir=pred_dir1)
                    te4a = time.time() - ts4a

                    if second_diameter is not None:
                        ts5b = time.time()
                        io.imread(str(im_path_s))
                        te5b = time.time() - ts5b

                        ts4b = time.time()
                        segmentation_helper.predict_single_image(
                            im_path_s,diameter=second_diameter,model=model,config=config,
                            save_masks=True,mute =True,model_id=model_id,tar_dir=pred_dir2)
                        te4b = time.time() - ts4b

                    #load images and prediction masks to combine them.
                    pred1 = os.path.join(pred_dir1, img_stem + f'_{model_id}_pred.tif')
                    pred2 = os.path.join(pred_dir2, img_stem + f'_{model_id}_pred.tif')
                    pred12 = os.path.join(pred_dir, img_stem + f'_{model_id}_combined_pred.tif')
                    if second_diameter is None:
                        ts4c = time.time()
                        shutil.copy(pred1, pred12)
                        te4c = time.time() - ts4c
                    else:
                        ts6 = time.time()
                        io.imread(str(pred1))
                        io.imread(str(pred2))
                        te6 = time.time() - ts6
                        
                        ts4d = time.time()
                        segmentation_helper.combine_preds(
                            [pred1],[pred2],[im_path_s],model_id=model_id,filters=filters,
                            threshold=threshold,tar_dir=pred_dir,do_composites=do_composites)
                        te4d = time.time() - ts4d

                    # Total predict time includes prediction at two scales, loading the predictions, and combining them.
                    te4 = te4a + te4b + te4c + te4d  
                    # Total image read time includes reading the image for prediction at two scales and 
                    # loading the predictions for combination.
                    te5 = te5a + te5b + te6

                    print(f"{img_file}, {i+1}/{npi}, image read time {te5:.2f} seconds, inference time {te4:.2f} seconds....")
                sts.append([img_file,foldername,model_id,model_id,sw,sh,te5, te4, -9999, -9999, -9999, -9999])

                ts4t = ts4t + te4
                ts5t = ts5t + te5
            print(f"{foldername} total image read time: {ts5t:.2f} seconds, total predict time: {ts4t:.2f} seconds")

            sts_df = pd.DataFrame(sts,columns=colnames)
            sts_df.to_csv(csvname,index=False)
    print('')
    sts_all = pd.concat([sts_all, sts_df], ignore_index=True)

sts_all_df = pd.DataFrame(sts_all,columns=colnames)
sts_all_df.to_csv(csvname_all,index=False)

te2 = time.time() - ts2
print(f"Total inference time for {len(img_paths)} folders: {te2:.2f} seconds")

Processing folder: FH


4_P1060348_3.jpg, 1/1, image read time 0.00 seconds, inference time 0.85 seconds....
FH total image read time: 0.00 seconds, total predict time: 0.85 seconds

Total inference time for 1 folders: 0.88 seconds


/tmp/ipykernel_4646/1224083610.py:120: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sts_all = pd.concat([sts_all, sts_df], ignore_index=True)


### 3b. Generating grain size and overlay images for all files in all folders.

In [6]:
lbl_key = 'segments_manual'
pred_key = 'segments_ig2'
img_paths2 = img_paths    # Only process the first folder for grain sizing and GSD calculation.
save_overlay = True
show_overlay = False
show_gsd = False
redraw_mask = True
column_name = 'ell: b-axis (px)'
properties=['label','area', 'area_convex','perimeter_crofton',
            'orientation','minor_axis_length','major_axis_length',
            'centroid','local_centroid','solidity','eccentricity','bbox']
percs = [16,50,84,96]

for folder_id in range(len(img_paths2)):
    print('=='*65)
    img_dir = img_paths2[folder_id]

    label_dir = os.path.join(img_dir,lbl_key)
    pred_dir = os.path.join(img_dir,pred_key)
    folder_name = os.path.basename(img_dir)
    img_files = natsorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png'))])
    npi = len(img_files)
    for img_id in range(npi):
        imgi_name = img_files[img_id]
        imgi_stem = os.path.splitext(imgi_name)[0]     
        imgi = os.path.join(img_dir, imgi_name)
        predi = os.path.join(pred_dir, imgi_stem + f'_{model_id}_combined_pred.tif')
        lbli = os.path.join(label_dir, f'{imgi_stem}_mask.tif')
        if not os.path.isfile(lbli): lbli = None

        # Checking if imagi is rgb, if not, skipping, imagi is an image path, not an array.
        # Loading image using imgi and then check its dimension.
        imagei = io.imread(imgi)
        if imagei.ndim != 3:
            print(f"{folder_name}: {imgi_name} ({img_id + 1}/{npi}) is not RGB, skipping...")
            continue

        print(f"{folder_name}: {imgi_name} ({img_id + 1}/{npi})")

        # Generating grain size from the segmentation masks.
        vis_files = []
        vis_files = [predi,lbli] if lbli is not None else [predi]
        for k in range(len(vis_files)):
            vis_file = vis_files[k]
            vis_dir = os.path.dirname(vis_file)
            saveflag = '' if vis_file.endswith('mask.tif') else f'{model_id}_'
            gsdcsv_name_i = f'{imgi_stem}_{saveflag}gsd.csv'
            gsdcsv_i = f'{vis_dir}/{gsdcsv_name_i}'
            saveoverlay_i = os.path.join(vis_dir,f'{imgi_stem}_{saveflag}overlay.jpg')
            saveoverlay_i2 = os.path.join(vis_dir,f'{imgi_stem}_{saveflag}overlay_ellipse.jpg')
            saveoverlay_i3 = os.path.join(vis_dir,f'{imgi_stem}_{saveflag}overlay_box.jpg')
            saveseg_i = os.path.join(vis_dir,f'{imgi_stem}_{saveflag}segmented.jpg')

            if os.path.isfile(gsdcsv_i) and os.path.isfile(saveoverlay_i) and not overwrite_plot:
                print(f"{gsdcsv_name_i} found, skipping grain sizing...")
                props_dfi = pd.read_csv(gsdcsv_i)
            else:
                # resolution = 390                      # resolution in um/px
                maski = io.imread(vis_file)
                props_dfi,propsi = grainsizing.grains_from_masks(maski,mute=True,filters=filters,properties=properties)

                # Write grain size data to file.
                props_dfi.to_csv(gsdcsv_i,index=False)

                #define plot elements
                elements= ['image','ellipse','ellipse_b','ellipse_a','bbox']
                elements2= ['image','ellipse','ellipse_b','ellipse_a']
                elements3= ['image','bbox']

                # use deterministic colors across repeated runs
                np.random.seed(42)

                # Color map control: 'tab20', 'tab20b', 'tab20c', 'Set3', 'Paired', 'nipy_spectral'
                #plt.rcParams['image.cmap'] = 'tab20'
                if redraw_mask:
                    # Use existing notebook variables
                    alpha = 0.2
                    msks = label2rgb(maski, image=imagei, bg_label=0, alpha=alpha, bg_color=None)
                    boundary = find_boundaries(maski, mode='outer')
                    msks[boundary] = (1.0, 0.0, 0.0)
                    msks_pil = Image.fromarray((np.clip(msks, 0, 1) * 255).astype(np.uint8))
                    msks_pil.save(saveseg_i)

                # plot and save image to disk.
                if save_overlay:
                    # Overlay with both ellipse and bbox.
                    plt.figure(figsize=(7,7))
                    plotting.all_grains_plot(maski,props=propsi,elements=elements,image=imagei,title=imgi_name,plot_padding=1000)
                    plotting.plot_single_img_mask(imagei,maski,bg_color=None,alpha=0.3)
                    ax = plt.gca()
                    ax.set_aspect("equal", adjustable="box")
                    ax.margins(0)
                    plt.savefig(saveoverlay_i, dpi=dpi, bbox_inches='tight')
                    if not show_overlay: plt.close()

                    plt.figure(figsize=(7,7))
                    plotting.all_grains_plot(maski,props=propsi,elements=elements2,image=imagei,title=imgi_name,plot_padding=1000)
                    plotting.plot_single_img_mask(imagei,maski,bg_color=None,alpha=0.3)
                    ax = plt.gca()
                    ax.set_aspect("equal", adjustable="box")
                    ax.margins(0)
                    plt.savefig(saveoverlay_i2, dpi=dpi, bbox_inches='tight')
                    if not show_overlay: plt.close()

                    plt.figure(figsize=(7,7))
                    plotting.all_grains_plot(maski,props=propsi,elements=elements3,image=imagei,title=imgi_name,plot_padding=1000)
                    plotting.plot_single_img_mask(imagei,maski,bg_color=None,alpha=0.3)
                    ax = plt.gca()
                    ax.set_aspect("equal", adjustable="box")
                    ax.margins(0)
                    plt.savefig(saveoverlay_i3, dpi=dpi, bbox_inches='tight')
                    if not show_overlay: plt.close()

            # calculate gsd and uncertainty
            sizes = props_dfi[column_name].values
            gsd_i = grainsizing.do_gsd(sizes)
            gsd_i = grainsizing.do_gsd(sizes)
            dx_i = grainsizing.get_key_percs(gsd_i,perc=percs)

            # Plot gsd_i vs pct_i = range(100)
            pct_i = np.arange(100)
            gsd_ii = np.array(gsd_i)

            # Set axis as pixel and y axis as Percent finer
            # Grid on;
            if show_gsd:
                plt.figure(figsize=(5,3))
                plt.plot(gsd_ii,pct_i,'-',color='b',markersize=4)
                plt.plot(dx_i,percs,'o',color='r',markersize=8)
                plt.xlabel('Grain size (px)')
                plt.ylabel('Percent finer (%)')
                plt.grid(True)
                plt.ylim(0,100)
                plt.title(f"{imgi_name}: {column_name}")
                plt.show()
    print('')

FH: 4_P1060348_3.jpg (1/1)
4_P1060348_3_IG2_full_set_cp_SAM_gsd.csv found, skipping grain sizing...



### 3c. Compute grain size distribution and uncertainty for all files and folders.

In [7]:
lbl_key =  'segments_manual'
pred_key = 'segments_ig2'

# The following cases do not have manual segmentation, so they are excluded from the summary analysis.
img_paths2 = img_paths.copy()  # Use copy to avoid modifying the original list
column_name = 'ell: b-axis (px)'
method = 'bootstrapping'
gsd_all = WorkDir + os.sep + 'Statistics_V3_' + parent_folder + '_' + model_id + '_' + computer + '.csv'

sdf = pd.DataFrame()
for folder_id in range(len(img_paths2)):
    img_dir = img_paths2[folder_id]
    folder_name = os.path.basename(img_dir)
    print(f"{folder_name}: ({folder_id + 1}/{len(img_paths2)})")
    vis_keys = [pred_key,lbl_key]
    outname = f"{folder_name}_summary_{method}_ellipse.csv"
    
    for vis_key in vis_keys:
        vis_dir = os.path.join(img_dir,vis_key)
        summary_csv_path = os.path.join(vis_dir, outname) 
        if os.path.isfile(summary_csv_path) and not overwrite_plot:
            print(f"{summary_csv_path} found, skipping...")
            summary_df = pd.read_csv(summary_csv_path)
        else:
            gsds = data_loader.gsds_from_folder(vis_dir,gsd_str='gsd')
            mask_str = 'mask' if vis_key == lbl_key else 'pred'
            if len(gsds) == 0: continue
            grains,grains_props,grain_id = grainsizing.batch_grainsize(vis_dir,mask_str=mask_str,mute=True,return_results=True,save_results=False)
            res_dict_bs = gsd_uncertainty.dataset_uncertainty(gsds=gsds,num_it=1000,method=method,\
                                                            mute=True,column_name=column_name,save_results=True,
                                                            return_results=True,sep=',',tar_dir=vis_dir,gsd_id=grain_id)
            summary_df = grainsizing.summary_statistics(gsds,grain_id,res_dict=res_dict_bs,unit='px',save_summary=True,data_id=folder_name)

        # Insert two columes to summary_df with 'Folder', and vis_key (pred vs manual).
        summary_df.insert(0, 'Folder', folder_name)
        summary_df.insert(1, 'Type', vis_key)
        sdf = pd.concat([sdf, summary_df], ignore_index=True)

sdf.sort_values(by=['Type', 'Folder'], inplace=True)
sdf.to_csv(gsd_all, index=False)
sdf.head()

FH: (1/1)
/qfs/people/chen200/Computation/Python/work/Mine/SFA/imagegrains/demo_data/FH/segments_ig2/FH_summary_bootstrapping_ellipse.csv found, skipping...


,Folder,Type,Image/Masks,number of grains,D16,CI D16 (95%),D50,CI D50 (95%),D84,CI D84 (95%),D96,CI D96 (95%),unit,axis,method,grain approximation
0,FH,segments_ig2,4_P1060348_3_IG2_full_set_cp_SAM_combined_pred,88,15.4,[14.2 17.2],22.1,[19.3 24.1],37.3,[29.3 51. ],68.0,[ 50.4 105.8],px,b-axis,bootstrapping,ellipse


In [8]:
te0 = time.time() - ts0
print(f"Total time for prediction: {te2:.2f} seconds")
print(f"Total time for full workflow: {te0:.2f} seconds")

Total time for prediction: 0.88 seconds
Total time for full workflow: 3.45 seconds
